# Official Replay Candidate Mechanism PK v1

Goal: aggregate recent official warm-start replay candidates across mechanisms and rank them with hard gate, robustness margin, and near-miss diagnostics before opening another training branch.

External basis:

- Qlib workflow/replay discipline: https://github.com/microsoft/qlib and https://arxiv.org/abs/2009.11189
- FinRL train/test/trade separation: https://github.com/AI4Finance-Foundation/FinRL and https://arxiv.org/abs/2011.09607
- Backtest-overfitting risk from repeated search: https://doi.org/10.21314/JCF.2016.322
- Volatility-managed portfolio intuition for state-conditioned risk, not unconditional hard caps: https://papers.ssrn.com/sol3/papers.cfm?abstract_id=2659431

This notebook is diagnostic only. Near-miss candidates are not promotion evidence; they identify the next robustness or reward-timing loop.


In [1]:

from __future__ import annotations

import json
import math
import os
import sys
from datetime import datetime, timezone
from pathlib import Path
from typing import Any

import pandas as pd
from IPython.display import Markdown, display

ROOT = Path.cwd()
while not (ROOT / "AGENTS.md").exists() and ROOT != ROOT.parent:
    ROOT = ROOT.parent
if not (ROOT / "AGENTS.md").exists():
    ROOT = Path("/Users/rziiiii/.openclaw/workspace")
os.chdir(ROOT)
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

from src.soxl_quant.app.official_candidate_aggregation import (
    OfficialReplayCandidateConfig,
    normalise_official_replay_candidate_frame,
    render_official_replay_candidate_review,
    score_official_replay_candidate_pool,
)

EXPERIMENT = "official_replay_candidate_mechanism_pk_v1"
CARD = ROOT / "runtime/model_selection/card" / EXPERIMENT
CARD.mkdir(parents=True, exist_ok=True)
CONFIG = OfficialReplayCandidateConfig(
    target_relative_return_pp=30.0,
    target_max_dd_pct=15.0,
    robust_relative_margin_pp=2.0,
    robust_max_dd_margin_pct=1.0,
    near_miss_relative_gap_pp=1.0,
    near_miss_max_dd_gap_pct=0.5,
)
EXTERNAL_SOURCES = [
    "https://github.com/microsoft/qlib",
    "https://arxiv.org/abs/2009.11189",
    "https://github.com/AI4Finance-Foundation/FinRL",
    "https://arxiv.org/abs/2011.09607",
    "https://doi.org/10.21314/JCF.2016.322",
    "https://papers.ssrn.com/sol3/papers.cfm?abstract_id=2659431",
]
SOURCES = [
    {
        "source_label": "bc_initialization_official",
        "mechanism": "bc_warm_start_baseline",
        "path": ROOT / "runtime/model_selection/card/short_volshock_bc_initialization_official_replay_v1/official_replay_metrics.csv",
        "candidate_col": "candidate",
    },
    {
        "source_label": "risk_state_target_screen",
        "mechanism": "hard_target_cap",
        "path": ROOT / "runtime/model_selection/card/official_replay_risk_state_target_screen_v1/candidate_pk.csv",
        "candidate_col": "candidate",
    },
    {
        "source_label": "short_volshock_sell_guard",
        "mechanism": "action_cap_overlay",
        "path": ROOT / "runtime/model_selection/card/short_volshock_sell_guard_official_replay_v1/official_replay_metrics.csv",
        "candidate_col": "variant",
        "reference_candidate_col": "candidate",
    },
]


def json_safe(value: Any) -> Any:
    if isinstance(value, dict):
        return {str(k): json_safe(v) for k, v in value.items()}
    if isinstance(value, (list, tuple)):
        return [json_safe(v) for v in value]
    if hasattr(value, "item"):
        try:
            return json_safe(value.item())
        except Exception:
            pass
    if isinstance(value, float):
        return value if math.isfinite(value) else None
    if isinstance(value, Path):
        return str(value)
    return value


def write_json(path: Path, payload: dict[str, Any]) -> None:
    path.write_text(json.dumps(json_safe(payload), indent=2, sort_keys=True, allow_nan=False) + "\n", encoding="utf-8")

print({"experiment": EXPERIMENT, "sources": [str(s["path"].relative_to(ROOT)) for s in SOURCES]})


{'experiment': 'official_replay_candidate_mechanism_pk_v1', 'sources': ['runtime/model_selection/card/short_volshock_bc_initialization_official_replay_v1/official_replay_metrics.csv', 'runtime/model_selection/card/official_replay_risk_state_target_screen_v1/candidate_pk.csv', 'runtime/model_selection/card/short_volshock_sell_guard_official_replay_v1/official_replay_metrics.csv']}


In [2]:

source_manifest = []
frames = []
for spec in SOURCES:
    path = spec["path"]
    raw = pd.read_csv(path)
    if "status" in raw.columns:
        raw = raw[raw["status"].astype(str).eq("evaluated")].copy()
    normalised = normalise_official_replay_candidate_frame(
        raw,
        source_label=str(spec["source_label"]),
        mechanism=str(spec["mechanism"]),
        candidate_col=str(spec.get("candidate_col", "")),
        config=CONFIG,
    )
    normalised["source_path"] = str(path.relative_to(ROOT))
    frames.append(normalised)
    source_manifest.append(
        {
            "source_label": spec["source_label"],
            "mechanism": spec["mechanism"],
            "path": str(path.relative_to(ROOT)),
            "rows": int(len(raw)),
            "candidates": int(normalised["candidate_id"].nunique()) if not normalised.empty else 0,
            "windows": sorted(normalised["window"].unique().tolist()) if not normalised.empty else [],
        }
    )

pool = pd.concat(frames, ignore_index=True) if frames else pd.DataFrame()
scored = score_official_replay_candidate_pool(pool, config=CONFIG)
review = render_official_replay_candidate_review(scored)
(CARD / "official_candidate_mechanism_pk.md").write_text(review, encoding="utf-8")
pool.to_csv(CARD / "official_candidate_pool.csv", index=False)
scored.to_csv(CARD / "mechanism_scores.csv", index=False)
pd.DataFrame(source_manifest).to_csv(CARD / "source_manifest.csv", index=False)

display(pd.DataFrame(source_manifest))
display(scored[["candidate_id", "mechanism", "window_count", "gate_pass_count", "full_gate", "robust_full_gate", "near_miss", "total_gate_loss", "min_relative_return", "max_max_dd", "failure_profile"]].head(20))
display(Markdown(review))


,source_label,mechanism,path,rows,candidates,windows
0,bc_initialization_official,bc_warm_start_baseline,runtime/model_selection/card/short_volshock_bc...,2,1,"[wf_2022, wf_2023_2024]"
1,risk_state_target_screen,hard_target_cap,runtime/model_selection/card/official_replay_r...,10,5,"[wf_2022, wf_2023_2024]"
2,short_volshock_sell_guard,action_cap_overlay,runtime/model_selection/card/short_volshock_se...,8,4,"[wf_2022, wf_2023_2024]"


,candidate_id,mechanism,window_count,gate_pass_count,full_gate,robust_full_gate,near_miss,total_gate_loss,min_relative_return,max_max_dd,failure_profile
0,short_volshock_sell_guard::short_volshock_cap0...,action_cap_overlay,2,1,False,False,True,1.256545,29.817263,15.107381,near_miss
1,short_volshock_sell_guard::short_volshock_cap0...,action_cap_overlay,2,1,False,False,False,5.155312,32.213744,15.515531,drawdown_control
2,short_volshock_sell_guard::short_volshock_cap0...,action_cap_overlay,2,1,False,False,False,5.179090,32.320004,15.517909,drawdown_control
3,risk_state_target_screen::volstress_all_target...,hard_target_cap,2,1,False,False,False,29.110326,2.002060,15.111239,trend_capture
4,risk_state_target_screen::volstress_all_target...,hard_target_cap,2,1,False,False,False,45.006679,-3.160067,16.184661,joint_tradeoff
5,bc_initialization_official::bc_initialization,bc_warm_start_baseline,2,0,False,False,False,114.531866,5.018899,22.261989,joint_tradeoff
6,risk_state_target_screen::bc_initialization,hard_target_cap,2,0,False,False,False,114.531866,5.018899,22.261989,joint_tradeoff
7,risk_state_target_screen::risk_state_all_targe...,hard_target_cap,2,0,False,False,False,114.531866,5.018899,22.261989,joint_tradeoff
8,risk_state_target_screen::risk_state_risk_targ...,hard_target_cap,2,0,False,False,False,114.531866,5.018899,22.261989,joint_tradeoff
9,short_volshock_sell_guard::baseline_no_overlay,action_cap_overlay,2,0,False,False,False,167.488279,30.262521,30.626799,drawdown_control


# Official Replay Candidate Mechanism PK

| candidate | mechanism | full | robust | near miss | loss | min rel | max dd | failure |
| --- | --- | --- | --- | --- | ---: | ---: | ---: | --- |
| short_volshock_sell_guard::short_volshock_cap05_hit2_no_guard | action_cap_overlay | False | False | True | 1.26 | 29.82pp | 15.11% | near_miss |
| short_volshock_sell_guard::short_volshock_cap05_hit2_sell_guard_zero | action_cap_overlay | False | False | False | 5.16 | 32.21pp | 15.52% | drawdown_control |
| short_volshock_sell_guard::short_volshock_cap05_hit2_sell_guard_1pct | action_cap_overlay | False | False | False | 5.18 | 32.32pp | 15.52% | drawdown_control |
| risk_state_target_screen::volstress_all_targets_cap010 | hard_target_cap | False | False | False | 29.11 | 2.00pp | 15.11% | trend_capture |
| risk_state_target_screen::volstress_all_targets_cap005 | hard_target_cap | False | False | False | 45.01 | -3.16pp | 16.18% | joint_tradeoff |
| bc_initialization_official::bc_initialization | bc_warm_start_baseline | False | False | False | 114.53 | 5.02pp | 22.26% | joint_tradeoff |
| risk_state_target_screen::bc_initialization | hard_target_cap | False | False | False | 114.53 | 5.02pp | 22.26% | joint_tradeoff |
| risk_state_target_screen::risk_state_all_targets_cap010 | hard_target_cap | False | False | False | 114.53 | 5.02pp | 22.26% | joint_tradeoff |
| risk_state_target_screen::risk_state_risk_target_cap010 | hard_target_cap | False | False | False | 114.53 | 5.02pp | 22.26% | joint_tradeoff |
| short_volshock_sell_guard::baseline_no_overlay | action_cap_overlay | False | False | False | 167.49 | 30.26pp | 30.63% | drawdown_control |

Near-miss candidates are diagnostic only. They require wider replay or stress testing before promotion.


In [3]:

mechanism_rows = []
for mechanism, sub in scored.groupby("mechanism", sort=True):
    best = sub.iloc[0]
    mechanism_rows.append(
        {
            "mechanism": mechanism,
            "candidate_count": int(len(sub)),
            "robust_full_gate_count": int(sub["robust_full_gate"].sum()),
            "full_gate_count": int(sub["full_gate"].sum()),
            "near_miss_count": int(sub["near_miss"].sum()),
            "best_candidate_id": str(best["candidate_id"]),
            "best_total_gate_loss": float(best["total_gate_loss"]),
            "best_min_relative_return": float(best["min_relative_return"]),
            "best_max_dd": float(best["max_max_dd"]),
            "best_failure_profile": str(best["failure_profile"]),
        }
    )
mechanism_summary = pd.DataFrame(mechanism_rows).sort_values(
    ["robust_full_gate_count", "full_gate_count", "near_miss_count", "best_total_gate_loss"],
    ascending=[False, False, False, True],
).reset_index(drop=True)
mechanism_summary.to_csv(CARD / "mechanism_summary.csv", index=False)

if scored.empty:
    decision = "no_official_replay_candidates_to_compare"
    selected_candidate = ""
    next_mechanism = "collect_official_replay_candidate_metrics"
    promotion_allowed = False
elif bool(scored.iloc[0]["robust_full_gate"]):
    decision = "robust_official_replay_candidate_found_no_auto_promotion"
    selected_candidate = str(scored.iloc[0]["candidate_id"])
    next_mechanism = "run wider fixed-split and walk-forward replay before actor retention"
    promotion_allowed = False
elif bool(scored.iloc[0]["full_gate"]):
    decision = "full_gate_candidate_has_low_margin_no_promotion"
    selected_candidate = str(scored.iloc[0]["candidate_id"])
    next_mechanism = "stress-test costs, split perturbations, and official replay before promotion"
    promotion_allowed = False
elif bool(scored.iloc[0]["near_miss"]):
    decision = "near_miss_action_cap_overlay_is_best_diagnostic_no_promotion"
    selected_candidate = str(scored.iloc[0]["candidate_id"])
    next_mechanism = "stress-test short-volshock action-cap overlay and convert only robust behavior into reward-timing objective"
    promotion_allowed = False
else:
    decision = "no_near_miss_official_candidate_no_promotion"
    selected_candidate = str(scored.iloc[0]["candidate_id"])
    next_mechanism = "derive training objective handoff before more PPO"
    promotion_allowed = False

conclusion = []
if not scored.empty:
    top = scored.iloc[0]
    conclusion.append(
        f"Best candidate is {top['candidate_id']} ({top['mechanism']}), full_gate={bool(top['full_gate'])}, "
        f"robust_full_gate={bool(top['robust_full_gate'])}, near_miss={bool(top['near_miss'])}, "
        f"min_rel={float(top['min_relative_return']):.2f}pp, max_dd={float(top['max_max_dd']):.2f}%, loss={float(top['total_gate_loss']):.2f}."
    )
for row in mechanism_summary.to_dict("records"):
    conclusion.append(
        f"Mechanism {row['mechanism']}: candidates={row['candidate_count']}, robust={row['robust_full_gate_count']}, "
        f"full={row['full_gate_count']}, near_miss={row['near_miss_count']}, best={row['best_candidate_id']}."
    )
conclusion.append("No candidate is promotable from this aggregation because no robust full-gate candidate exists.")
conclusion.append(f"Next mechanism: {next_mechanism}.")

selection = {
    "experiment": EXPERIMENT,
    "created_at": datetime.now(timezone.utc).isoformat(),
    "decision": decision,
    "role": "diagnose",
    "canonical_baseline": "v19reg_v22_budget_usage_bull_guard_v1",
    "direct_comparison_baseline": "short_volshock_bc_initialization_official_replay_v1/bc_initialization",
    "changed_layer": ["model_selection", "official_replay_candidate_aggregation"],
    "external_reference": EXTERNAL_SOURCES,
    "hypothesis": "Recent official replay failures may contain a stable near-miss mechanism; aggregate candidates with safety margins before starting another reward/PPO branch.",
    "acceptance_criteria": {
        "hard_gate": ">=30pp relative and <=15% MaxDD in every supplied official replay window",
        "robust_gate": ">=32pp relative and <=14% MaxDD in every supplied official replay window",
    },
    "accepted_tradeoff": "Near-miss candidates are diagnostic only and cannot be promoted without robustness replay.",
    "failure_mode": "Repeatedly tuning around a <1pp/<0.5% near miss can overfit the replay window.",
    "closest_historical_success_case": "short_volshock_sell_guard_official_replay_v1 found an action-cap overlay near the official gate on both windows.",
    "closest_historical_failure_case": "official_replay_risk_state_target_screen_v1 showed hard target caps reduced drawdown in 2022 but destroyed 2024 excess.",
    "duplicate_check": "This run does not introduce a new threshold or PPO branch; it aggregates existing official replay candidates and applies robustness margins.",
    "selected_candidate": selected_candidate,
    "promotion_allowed": promotion_allowed,
    "next_mechanism": next_mechanism,
    "sidecars": [
        "source_manifest.csv",
        "official_candidate_pool.csv",
        "mechanism_scores.csv",
        "mechanism_summary.csv",
        "official_candidate_mechanism_pk.md",
        "selection.json",
        "summary.json",
    ],
}
summary = {
    "experiment": EXPERIMENT,
    "created_at": selection["created_at"],
    "decision": decision,
    "promotion_allowed": promotion_allowed,
    "source_manifest": source_manifest,
    "mechanism_summary": mechanism_summary.to_dict("records"),
    "top_candidates": scored.head(20).to_dict("records"),
    "conclusion": conclusion,
    "next_mechanism": next_mechanism,
    "external_sources": EXTERNAL_SOURCES,
    "sidecars": selection["sidecars"],
}
write_json(CARD / "selection.json", selection)
write_json(CARD / "summary.json", summary)

display(mechanism_summary)
print(json.dumps(json_safe(selection), indent=2, allow_nan=False))


,mechanism,candidate_count,robust_full_gate_count,full_gate_count,near_miss_count,best_candidate_id,best_total_gate_loss,best_min_relative_return,best_max_dd,best_failure_profile
0,action_cap_overlay,4,0,0,1,short_volshock_sell_guard::short_volshock_cap0...,1.256545,29.817263,15.107381,near_miss
1,hard_target_cap,5,0,0,0,risk_state_target_screen::volstress_all_target...,29.110326,2.002060,15.111239,trend_capture
2,bc_warm_start_baseline,1,0,0,0,bc_initialization_official::bc_initialization,114.531866,5.018899,22.261989,joint_tradeoff


{
  "experiment": "official_replay_candidate_mechanism_pk_v1",
  "created_at": "2026-05-26T18:47:55.753167+00:00",
  "decision": "near_miss_action_cap_overlay_is_best_diagnostic_no_promotion",
  "role": "diagnose",
  "canonical_baseline": "v19reg_v22_budget_usage_bull_guard_v1",
  "direct_comparison_baseline": "short_volshock_bc_initialization_official_replay_v1/bc_initialization",
  "changed_layer": [
    "model_selection",
    "official_replay_candidate_aggregation"
  ],
  "external_reference": [
    "https://github.com/microsoft/qlib",
    "https://arxiv.org/abs/2009.11189",
    "https://github.com/AI4Finance-Foundation/FinRL",
    "https://arxiv.org/abs/2011.09607",
    "https://doi.org/10.21314/JCF.2016.322",
    "https://papers.ssrn.com/sol3/papers.cfm?abstract_id=2659431"
  ],
  "hypothesis": "Recent official replay failures may contain a stable near-miss mechanism; aggregate candidates with safety margins before starting another reward/PPO branch.",
  "acceptance_criteria": {
 

## Runtime Conclusion

This notebook keeps the single-run conclusion local to this runtime card. The machine-readable conclusion is in `selection.json` and `summary.json`; the human-readable candidate table is in `official_candidate_mechanism_pk.md`.
